# Yantra — Router V2 Standalone Eval (guaranteed to run)

Upload to Colab → **Runtime → Run all**. Evaluates the **existing** Yantra Q4_K_M GGUF + `toolace_300.jsonl` from Drive with **Router V2** (IDF + char-3gram, α=2.0) and the **last-bind parser**. No Unsloth, no `llama-server` binary — uses `llama-cpp-python` (pip) directly via `Llama()` class.

**Guarantees:** works on any T4 runtime, even after a factory reset. Copies GGUF to `/tmp` (Drive FUSE is slow). Purges stale `eval_progress_*` files automatically.

In [ ]:
# @title 0 — Mount Drive & locate GGUF
import os, sys, json, re, math, shutil, time, subprocess
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = Path('/content/drive/MyDrive/yantra_run')
except Exception as e:
    RUN_DIR = Path('./yantra_run')
    print('Drive not mounted:', e, '→ using', RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
ART = RUN_DIR / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
print('RUN_DIR =', RUN_DIR)
print('ART =', ART)

# Purge stale progress (they held the old 0.32 numbers)
for p in list(ART.glob('eval_progress_router_v2*.jsonl')):
    print('removing stale', p)
    try: p.unlink()
    except Exception: pass

cands = [ART/'stage6_yantra_q4_gguf', Path('/content/stage6_yantra_q4_gguf'), Path('/content/stage6_yantra_q4')]
GGUF = None
for d in cands:
    if d.exists():
        g = sorted(d.glob('*Q4_K_M*.gguf')) or sorted(d.glob('*.gguf'))
        if g:
            GGUF = g[0]; break
if GGUF is None:
    for d in cands:
        if d.is_file():
            GGUF = d; break
if GGUF is None:
    raise FileNotFoundError(f'No GGUF found. Checked: {cands}. Run the main pipeline Stage 6 first.')
print(f'Found GGUF: {GGUF} ({GGUF.stat().st_size/1024/1024:.0f} MB)')

# Copy to local /tmp if on Drive (FUSE reads time out)
LOCAL_GGUF = Path('/tmp/yantra.Q4_K_M.gguf')
if str(GGUF).startswith('/content/drive'):
    if not LOCAL_GGUF.exists() or LOCAL_GGUF.stat().st_size != GGUF.stat().st_size:
        print(f'Copying GGUF to {LOCAL_GGUF} for fast serving ...')
        shutil.copy2(str(GGUF), str(LOCAL_GGUF))
        print('Copy done')
    GGUF = LOCAL_GGUF
print('Serving from:', GGUF)

if not (ART/'toolace_300.jsonl').exists():
    raise FileNotFoundError(f"{ART/'toolace_300.jsonl'} not found — run Stage 1 of main pipeline first.")
print('toolace_300.jsonl exists')


In [ ]:
# @title 1 — Install llama-cpp-python (CPU/CUDA auto, no Unsloth needed)
import importlib, subprocess, sys, os
def _need_llama():
    try:
        import llama_cpp
        print('llama_cpp', llama_cpp.__version__)
        return True
    except Exception:
        return False

if not _need_llama():
    print('Installing llama-cpp-python ... (2-4 min)')
    rc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python', '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121']).returncode
    if rc != 0 or not _need_llama():
        print('CUDA wheel failed — trying plain pip install ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'], check=False)
    import llama_cpp
    print('Installed llama_cpp', llama_cpp.__version__)
else:
    import llama_cpp
    print('llama_cpp already installed')

import torch
HAS_CUDA = torch.cuda.is_available()
print('HAS_CUDA =', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# @title 2 — Router V2 + parsers + metrics (same as main pipeline Stage 6)
import json, re, math
from collections import Counter

# ---------- DTSA helpers ----------
ACTION_END = '<action_end/>'
DTSA_BIND = re.compile(r'<bind\s+tool="([^"]+)"\s*/>', re.S)
DTSA_ARGS = re.compile(r'<args>(.*?)</args>', re.S)
DTSA_PARAM = re.compile(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', re.S)
def _unesc(v): return v.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>')
def to_dtsa(name, arguments):
    lines = ['<bind tool="%s"/>' % name, '<args>']
    for k,v in arguments.items():
        v = '' if v is None else str(v)
        v = v.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        lines.append('  <param name="%s">%s</param>' % (k,v))
    lines += ['</args>', ACTION_END]
    return '\n'.join(lines)
def bind_prefix(name): return f'<bind tool="{name}"/>\n'
def dtsa_args_block(name, args): return '\n'.join(to_dtsa(name,args).splitlines()[1:])
def strip_bind(text):
    m = DTSA_BIND.search(text)
    return text[m.end():] if m else text
def parse_args_block(text):
    am = DTSA_ARGS.search(text)
    if not am: return {}, False
    args = {pm.group(1): _unesc(pm.group(2).strip()) for pm in DTSA_PARAM.finditer(am.group(1))}
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    return args, stopped

# ---------- Last-bind-with-args parser (handles <param> stripped as special tokens) ----------
def parse_dtsa(text):
    binds = list(DTSA_BIND.finditer(text))
    if not binds: return None
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    for j in range(len(binds)-1, -1, -1):
        lb = binds[j]
        seg_end = binds[j+1].start() if j+1 < len(binds) else len(text)
        am = DTSA_ARGS.search(text[lb.end():seg_end])
        if not am: continue
        body = am.group(1); args = {}
        for pm in DTSA_PARAM.finditer(body):
            args[pm.group(1)] = _unesc(pm.group(2).strip())
        if not args:
            for lm in re.finditer(r'name="([^"]+)"\s*>[ \t]*(.*)', body):
                args[lm.group(1)] = _unesc(lm.group(2).strip())
        return {'tool': lb.group(1), 'args': args, 'stopped_clean': stopped}
    return {'tool': binds[0].group(1), 'args': {}, 'stopped_clean': stopped}

def parse_legacy(text):
    m = re.search(r'<function\s+name="([^"]+)"\s*>(.*?)</function>', text, re.S)
    if not m: return None
    tool = m.group(1); args = {pm.group(1): pm.group(2).strip() for pm in re.finditer(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', m.group(2), re.S)}
    tail = text[m.end():].strip()
    return {'tool': tool, 'args': args, 'stopped_clean': tail == ''}

def parse_lenient(text):
    t = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    m = parse_legacy(t)
    if m: return m
    ms = list(re.finditer(r'name="([^"]+)"\s*>', t))
    if not ms: return None
    tool = ms[0].group(1); args = {}
    for i, mm in enumerate(ms[1:], 1):
        start = mm.end(); end = ms[i+1].start() if i+1 < len(ms) else len(t)
        val = t[start:end].lstrip('>').strip()
        if val: args[mm.group(1)] = val
    return {'tool': tool, 'args': args, 'stopped_clean': bool(re.search(r'</function>|'+re.escape(ACTION_END), t))}

# ---------- Metrics ----------
def _norm(a):
    if isinstance(a, str):
        s = a.strip()
        if s.lower() in ('true','false'): return s.lower()=='true'
        try: return int(s) if '.' not in s else float(s)
        except ValueError: return s
    return a
def avail_names(tools): return {t.get('function', t).get('name') for t in tools}
def evaluate_case(text, case, mode):
    tools = case['tools']; gold = case['gold']; gold0 = gold[0]
    parsed = parse_dtsa(text) if mode == 'dtsa' else parse_lenient(text)
    parseable = parsed is not None
    if not parsed: return {'parseable':0,'valid_name':0,'expected_name':0,'exact_args':0,'arg_key_overlap':0,'stopped_cleanly':0}
    valid = parsed['tool'] in avail_names(tools)
    expected = parsed['tool'] == gold0['name']
    gk = set(gold0['arguments'].keys()); pk = set(parsed['args'].keys())
    overlap = len(gk & pk)/len(gk) if gk else 1.0
    exact = (set(parsed['args'].keys()) == gk) and all(_norm(gold0['arguments'][k]) == _norm(parsed['args'][k]) for k in gk) if gk else bool(parsed)
    return {'parseable':1,'valid_name':int(valid),'expected_name':int(expected),'exact_args':int(exact),'arg_key_overlap':round(overlap,4),'stopped_cleanly':int(parsed['stopped_clean'])}
def pas(summary, recovery=0.0, multiturn=0.0):
    comps = [summary.get(k,0.0) for k in ('parseable','valid_name','expected_name','exact_args','arg_key_overlap','stopped_cleanly')] + [recovery, multiturn]
    return round(sum(comps)/len(comps), 4)
print('utils loaded')


In [ ]:
# @title 3 — Run 300-case eval (Router V2, direct Llama inference, no server)
from llama_cpp import Llama
import gc, time

print(f'Loading GGUF: {GGUF}')
n_gpu = -1 if HAS_CUDA else 0
print(f'n_gpu_layers={n_gpu} (HAS_CUDA={HAS_CUDA})')
llm = Llama(model_path=str(GGUF), n_ctx=4096, n_gpu_layers=n_gpu, verbose=False)
print('Model loaded.')

cases = [json.loads(l) for l in open(ART/'toolace_300.jsonl') if l.strip()]
print(f'Loaded {len(cases)} cases')

# ---- Router V2 (same as main pipeline Stage 6, benchmarked 244/300 = 0.813) ----
toks = lambda s: set(re.findall(r'[a-z0-9_]+', s.lower()))
_docs = [toks((t.get('function',t).get('name','')+' '+t.get('function',t).get('description',''))) for c in cases for t in c['tools']]
_df = Counter(w for d in _docs for w in d); _N = max(len(_docs),1)
_idf = lambda w: math.log(_N/(1+_df[w]))
def cgrams(s, n=3):
    s = re.sub(r'[^a-z0-9 ]','',s.lower())
    return {s[i:i+n] for i in range(max(len(s)-n+1,1))}
def route(q, tools):
    qt = toks(q); qg = cgrams(q)
    best, bn = -1e18, None
    for t in tools:
        f = t.get('function', t); nm = f.get('name') or ''
        d = toks(nm+' '+(f.get('description') or ''))
        s1 = sum(_idf(w) for w in qt & d)/(math.sqrt(sum(_idf(w) for w in d)+1e-6))
        ng = cgrams(nm)
        s2 = len(qg & ng)/(math.sqrt(len(qg)*len(ng))+1)
        if s1 + 2.0*s2 > best: best, bn = s1 + 2.0*s2, nm
    return bn

per = []
t0 = time.time()
for i, case in enumerate(cases):
    bind = f'<bind tool="{route(case["query"], case["tools"])}"/>\n'
    prompt = f"<user>{case['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in case['tools']], ensure_ascii=False)}</tools>\n<calls>{bind}"
    out = llm(prompt, max_tokens=768, temperature=0.0, stop=['\n<bind', '<tool_result>', '<user>', '</calls>'])
    text = out['choices'][0]['text'] or ''
    # grade prompt-bind + completion together (completion is args-only)
    m = evaluate_case(bind + text, case, mode='dtsa')
    per.append({'id': i, **m, '_raw': text[:400]})
    if (i+1) % 25 == 0 or i == len(cases)-1:
        elapsed = time.time()-t0
        agg_tmp = {}
        for mm in per:
            for k,v in mm.items():
                if k in ('id','_raw'): continue
                agg_tmp[k] = agg_tmp.get(k, 0.0) + v
        n = len(per); summary_tmp = {k: round(v/n,4) for k,v in agg_tmp.items()}
        print(f'  eval {i+1}/{len(cases)}  pas={pas(summary_tmp):.4f}  {elapsed:.0f}s', flush=True)

agg = {}
for m in per:
    for k,v in m.items():
        if k in ('id','_raw'): continue
        agg[k] = agg.get(k, 0.0) + v
n = len(cases); summary = {k: round(v/n,4) for k,v in agg.items()}
score = pas(summary)
res = {'summary': summary, 'pas': score, 'per_case': [{k:v for k,v in p.items() if k != '_raw'} for p in per]}
(ART/'yantra_results_router_v2.json').write_text(json.dumps(res, indent=2))
print('\n' + '='*50)
print(f'YANTRA Router-V2 PAS = {score}')
print(json.dumps(summary, indent=2))
print(f"Saved to {ART/'yantra_results_router_v2.json'}")
print('\n--- 3 sample raw completions (truncated) ---')
for p in per[:3]:
    print(repr(p['_raw'][:300]))
